## Structured Output Parser

structured output parser is a output parser in langchain that helps extract structured JSON data from LLM based on 
predefined field schemas.

It works by defining a list of fields that the model should return , ensuring the output follows a Structured format

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser          
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

import langchain



load_dotenv()

model = ChatGoogleGenerativeAI (
    model="gemini-2.5-flash",
    temperature=0,
)

c:\Users\KIIT0001\Desktop\Langchain-from-Scratch\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
schema = [
    ResponseSchema(name='fact_1' , description='Fact 1 about the topic'),
    ResponseSchema(name='fact_2' , description='Fact 2 about the topic'),
    ResponseSchema(name='fact_3' , description='Fact 3 about the topic'),
]

In [3]:
parser = StructuredOutputParser.from_response_schemas(schema)

In [4]:
template = PromptTemplate(
    template='Give 3 fact about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

In [5]:
prompt = template.invoke({'topic' : 'black hole'}) 

In [6]:
result = model.invoke(prompt)

In [9]:
print(result)

content='```json\n{\n\t"fact_1": "Black holes are regions in spacetime where gravity is so strong that nothing, not even light, can escape once it crosses a certain boundary called the event horizon.",\n\t"fact_2": "Most black holes form from the remnants of massive stars that collapse under their own gravity at the end of their life cycle, but supermassive black holes, millions to billions of times the mass of our Sun, reside at the centers of most galaxies.",\n\t"fact_3": "Despite their name, black holes are not \'holes\' in space; they are incredibly dense objects where a huge amount of matter is packed into a very small area, distorting spacetime around them."\n}\n```' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cb924-f30f-7ff2-80a8-1283eb7e48f6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 94, 'output_tokens': 427, 'total_tokens': 5

In [10]:
final_result = parser.parse(result.content)

In [13]:
print(final_result)

{'fact_1': 'Black holes are regions in spacetime where gravity is so strong that nothing, not even light, can escape once it crosses a certain boundary called the event horizon.', 'fact_2': 'Most black holes form from the remnants of massive stars that collapse under their own gravity at the end of their life cycle, but supermassive black holes, millions to billions of times the mass of our Sun, reside at the centers of most galaxies.', 'fact_3': "Despite their name, black holes are not 'holes' in space; they are incredibly dense objects where a huge amount of matter is packed into a very small area, distorting spacetime around them."}


In [14]:
import json
print(json.dumps(final_result, indent=2))


{
  "fact_1": "Black holes are regions in spacetime where gravity is so strong that nothing, not even light, can escape once it crosses a certain boundary called the event horizon.",
  "fact_2": "Most black holes form from the remnants of massive stars that collapse under their own gravity at the end of their life cycle, but supermassive black holes, millions to billions of times the mass of our Sun, reside at the centers of most galaxies.",
  "fact_3": "Despite their name, black holes are not 'holes' in space; they are incredibly dense objects where a huge amount of matter is packed into a very small area, distorting spacetime around them."
}


## Another way using CHAINS

In [15]:
chain = template | model | parser

In [16]:
result = chain.invoke({'topic' : 'black hole'})

In [18]:
print(json.dumps(result, indent=2))

{
  "fact_1": "Black holes are regions in spacetime where gravity is so strong that nothing, not even light, can escape once it crosses a certain boundary called the event horizon.",
  "fact_2": "Most black holes form from the remnants of massive stars that collapse under their own gravity at the end of their life cycle, but supermassive black holes, millions to billions of times the mass of our Sun, reside at the centers of most galaxies.",
  "fact_3": "Despite their name, black holes are not 'holes' in space; they are incredibly dense objects where a huge amount of matter is packed into a very small area, distorting spacetime around them."
}


## Disadvantages of structured output parsers 

is that we cannot perform data validations , we can only determine the structuer of the output ....suppose we wanted the name to be a string , age to an integer an city to be a string .....but it returned age as 22 years now this is a string!! 

so either we have to accept that or process them manually. 

Thats why we use pydantic output parser.